# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: "What Predicts Health?" (Random Forest, p.27)

**Claim:** Average Position (43%), Impressions (32%), and Scroll Depth (15%) are the top predictors of a page's Health Score.

**Where does the label come from?**
Health Score is FlyRank's own composite metric, defined in the paper's methodology as: Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts). It is not an independent outcome — it's built directly out of several of the same columns used as model features.

**Does the validation design carry the claim?**
No, their stated method (an `80/20 train/test split`) can't fix this. This is label-derived-feature leakage and not a splitting problem: Average Position, Impressions, and CTR are literally components of the formula used to compute the label. Holding out rows doesn't help, because the leak lives in the *formula*, meaning it is completely irrelevant from a train/test split as any split would still show the same inflated importance. The paper is honest about this itself in its statement it said: ("the target is partly constructed from some of these inputs, so importance is descriptive rather than causal"), but the paper never runs a with/without test to show how much of the 43-32-15 split is definitional versus genuine signal.

**My methodology question:** what would feature importance look like with Position, Impressions, and CTR removed, re-run on only the remaining independent features? That's the train-without-suspect test from the leakage checklist, and it isn't shown here.

---

### Finding 2: "What Predicts Growth?" (Logistic Regression, p.29)

**Claim:** 71% holdout accuracy predicting growing vs. declining pages, with Content Age as the strongest negative signal.

**Where does the label come from?**
Growth/decline direction is defined elsewhere in the paper (Finding #1) as 30-day-vs-previous-30-day impression change (up = >10% growth, down = >10% decline). Comparable to my own label design — reasonably clean, no obvious label-derived features in the coefficient list.

**Does the validation design carry the claim?**
Partially, in two ways:
1. The methodology states an 80/20 split with no mention of grouping by brand. This portfolio spans 57 brands; if pages from the same brand land on both sides of a random split, the model could partly be learning brand-level patterns rather than a generalizable rule — the exact client-grouping problem I fixed in my own Week 5 model.
2. Base rate check: Finding #1 reports ~74.8K growing vs. ~45.6K declining pages portfolio-wide — a base rate near 62% for "growing." Always predicting "growing" would already score ~62% accuracy. 71% is only about 9 points above that naive baseline, not 71 points of skill — and the paper doesn't state the base rate next to the 71% figure.

**My methodology question:** was the split grouped by brand, and what was the base rate of the "growing" class in the holdout set specifically? Without both numbers, 71% accuracy is hard to interpret honestly.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    cache_file = candidate / 'work' / 'outputs' / 'february_march_features.parquet'
    if cache_file.exists():
        repo_root = candidate
        break

cache_path = repo_root / 'work' / 'outputs' / 'february_march_features.parquet'
dataframe = pd.read_parquet(cache_path)
dataframe = dataframe[dataframe['future_decline_label'].notna()].copy()

# Same feature prep as Week 5
X = dataframe[['prior_impressions', 'prior_clicks', 'prior_avg_position',
               'prior_sessions', 'prior_engagement_rate']].copy()
X['prior_avg_position'] = X['prior_avg_position'].fillna(999)
X['prior_sessions'] = X['prior_sessions'].fillna(0)
X['prior_engagement_rate'] = X['prior_engagement_rate'].fillna(0)
X['prior_ctr'] = (dataframe['prior_clicks'] / dataframe['prior_impressions'].replace(0, np.nan)).fillna(0)

model_features = ['prior_impressions', 'prior_clicks', 'prior_ctr',
                   'prior_avg_position', 'prior_sessions', 'prior_engagement_rate']
X = X[model_features]
y = dataframe['future_decline_label'].values
groups = dataframe['client_hash_id']

def precision_at_20(y_true, scores):
    order = np.argsort(-scores)
    return y_true[order][:20].mean()

def fit_and_score(X_train, X_test, y_train, y_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    model = LogisticRegression(random_state=42, max_iter=1000, solver='lbfgs')
    model.fit(X_train_scaled, y_train)
    scores = model.predict_proba(X_test_scaled)[:, 1]
    return {
        'test_rows': len(y_test),
        'base_rate': float(y_test.mean()),
        'precision_at_20': precision_at_20(y_test, scores),
    }

# BEFORE: current approach — fit AND evaluate on all rows (in-sample, no held-out test)
before_result = fit_and_score(X, X, y, y)

# AFTER: honest split — grouped by client so no client appears on both sides
splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
after_result = fit_and_score(X.iloc[train_idx], X.iloc[test_idx], y[train_idx], y[test_idx])

overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f"Client overlap in the honest split: {len(overlap)} (should be 0)")

comparison = pd.DataFrame([
    {'split': 'before_in_sample_no_split', **before_result},
    {'split': 'after_grouped_by_client', **after_result},
])
comparison['gap_vs_before'] = comparison['precision_at_20'] - before_result['precision_at_20']
comparison

Client overlap in the honest split: 0 (should be 0)


,split,test_rows,base_rate,precision_at_20,gap_vs_before
0,before_in_sample_no_split,80322,0.217549,0.45,0.0
1,after_grouped_by_client,28904,0.187690,0.35,-0.1


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
print("="*60)
print("TEST 1: TIMELINE CHECK")
print("="*60)
non_prior_features = [f for f in model_features if not f.startswith('prior_')]
print(f"Model features: {model_features}")
print(f"Features NOT starting with 'prior_': {non_prior_features}")
print("PASS: all features are prior-period" if len(non_prior_features) == 0 else "FAIL: found a non-prior feature")

print("\n" + "="*60)
print("TEST 2: LABEL-DERIVED FEATURE TEST (deliberately plant a leak)")
print("="*60)

# Deliberately add an obviously leaky feature built directly from the
# future column used to compute the label — to prove the test harness
# actually reacts to leakage, not just that it looks clean.
X_leaky = X.copy()
X_leaky['leak_future_ratio'] = (
    dataframe['future_impressions'] / dataframe['prior_impressions'].replace(0, np.nan)
).fillna(0)

leaky_result = fit_and_score(
    X_leaky.iloc[train_idx], X_leaky.iloc[test_idx], y[train_idx], y[test_idx]
)

print(f"Honest model (no leak), precision@20: {after_result['precision_at_20']:.3f}")
print(f"Model WITH planted leak, precision@20: {leaky_result['precision_at_20']:.3f}")
jump = leaky_result['precision_at_20'] - after_result['precision_at_20']
print(f"Jump: {jump:+.3f}")
print("This confirms the test harness reacts to leakage — seeing the score jump")
print("when I plant a leak, then fall back to the honest number once I remove it,")
print("is what proves the harness itself can be trusted.")

print("\n" + "="*60)
print("TEST 3: PRODUCT-FLAG / EXISTING-SYSTEM-SCORE CHECK")
print("="*60)
product_flag_columns = [
    'baseline_score', 'reason_code', 'high_visibility_at_risk',
    'weak_position_signal', 'low_prior_engagement',
    'low_click_through_rate', 'limited_prior_visibility',
]
used_flags = [c for c in product_flag_columns if c in model_features]
print(f"Baseline/product-flag columns found in model_features: {used_flags}")
print("PASS: no baseline/product-flag columns used as model inputs" if len(used_flags) == 0 else "FAIL")

print("\n" + "="*60)
print("TEST 4: FEATURE IMPORTANCE SANITY CHECK (honest model)")
print("="*60)
scaler_check = StandardScaler()
Xtr_scaled = scaler_check.fit_transform(X.iloc[train_idx])
Xte_scaled = scaler_check.transform(X.iloc[test_idx])
honest_model = LogisticRegression(random_state=42, max_iter=1000, solver='lbfgs')
honest_model.fit(Xtr_scaled, y[train_idx])

importance = pd.DataFrame({
    'feature': model_features,
    'abs_coefficient': np.abs(honest_model.coef_[0]),
}).sort_values('abs_coefficient', ascending=False)
print(importance.to_string(index=False))

dominant_share = importance['abs_coefficient'].iloc[0] / importance['abs_coefficient'].sum()
print(f"\nTop feature share of total coefficient weight: {dominant_share:.1%}")
print("Investigate further — one feature dominates" if dominant_share > 0.6 else "No single feature dominates — looks reasonable")

print("\n" + "="*60)
print("LEAKAGE AUDIT SUMMARY")
print("="*60)
audit_summary = pd.DataFrame({
    'check': [
        'all_features_are_prior_period',
        'planted_leak_detected_by_harness',
        'no_product_flags_in_features',
        'no_single_dominant_feature',
        'split_grouped_by_client_zero_overlap',
        'base_rate_reported_next_to_metric',
    ],
    'result': [
        len(non_prior_features) == 0,
        jump > 0.1,
        len(used_flags) == 0,
        dominant_share <= 0.6,
        len(overlap) == 0,
        True,  # confirmed in Section 2's comparison table
    ],
})
audit_summary

TEST 1: TIMELINE CHECK
Model features: ['prior_impressions', 'prior_clicks', 'prior_ctr', 'prior_avg_position', 'prior_sessions', 'prior_engagement_rate']
Features NOT starting with 'prior_': []
PASS: all features are prior-period

TEST 2: LABEL-DERIVED FEATURE TEST (deliberately plant a leak)
Honest model (no leak), precision@20: 0.350
Model WITH planted leak, precision@20: 1.000
Jump: +0.650
This confirms the test harness reacts to leakage — seeing the score jump
when I plant a leak, then fall back to the honest number once I remove it,
is what proves the harness itself can be trusted.

TEST 3: PRODUCT-FLAG / EXISTING-SYSTEM-SCORE CHECK
Baseline/product-flag columns found in model_features: []
PASS: no baseline/product-flag columns used as model inputs

TEST 4: FEATURE IMPORTANCE SANITY CHECK (honest model)
              feature  abs_coefficient
            prior_ctr         0.196053
       prior_sessions         0.144199
    prior_impressions         0.118402
prior_engagement_rate  

,check,result
0,all_features_are_prior_period,True
1,planted_leak_detected_by_harness,True
2,no_product_flags_in_features,True
3,no_single_dominant_feature,True
4,split_grouped_by_client_zero_overlap,True
5,base_rate_reported_next_to_metric,True


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.